# Katip Mask Base, clean · the Apache-2.0 candidate

**What this does.** Trains Base again **without Turkish Wiki NER** (the CC BY-SA data that forces our models to be
CC BY-SA). Instead it learns from court decisions up to 2023 (not copyrighted, FSEK art. 31) with refilled names, plus
Katip's synthetic documents. If KVKK-Bench stays within ~1 point of Base v2, v3 can be released under Apache-2.0.

| | |
|---|---|
| Time | ~5 min smoke test, then ~40-60 min on a free T4 |
| Needs | `train.jsonl` from the court-decision crawl. It stays on your Mac (`katip-legal-study/lab/data/legal_web/train.jsonl`) and is deliberately not synced anywhere. Upload it yourself in the left panel (Files) for this run only |
| Output | `katip-mask-base-clean.zip` |

If the run on your Mac has already finished, you don't need this notebook.

In [ ]:
# 1 · Check the GPU, get the code. Run this cell first. (Takes ~1 min.)
import os, subprocess, sys
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE. In Colab: Runtime > Change runtime type > T4 GPU. In Kaggle: Settings > Accelerator > GPU T4.")
if not os.path.exists("katip"):
    subprocess.run("git clone -q --depth 1 -b legal-study https://github.com/fikrifirat/katip", shell=True, check=True)
os.chdir("katip")
subprocess.run(f"{sys.executable} -m pip install -q regex tokenizers onnx onnxruntime faker huggingface_hub 'transformers>=4.44' accelerate", shell=True, check=True)
print(subprocess.run("git log --oneline -1", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# Optional · Hugging Face token, only needed to upload at the end.
# Colab: the key icon on the left > add a secret named HF_TOKEN (write access) and allow this notebook.
# Kaggle: Add-ons > Secrets > HF_TOKEN. Nothing is hard-coded here.
HF_TOKEN = None
try:
    from google.colab import userdata; HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
print("HF token:", "found" if HF_TOKEN else "none (fine: you can download the zip instead)")

In [ ]:
# Upload train.jsonl to /content first (left panel > Files > upload). Colab deletes it when the session ends.
import os
LEGAL = None
for p in ["/content/train.jsonl", "/kaggle/input/katip-legal-web/train.jsonl"]:
    if os.path.exists(p): LEGAL = p
print("court decisions:", LEGAL or "NOT FOUND: upload train.jsonl and run this cell again")

## 2 · Smoke test

In [ ]:
!cd ml/berturk && python train_berturk.py --legal "$LEGAL" --limit 300 --epochs 1 --out out_smoke 2>&1 | tail -8

## 3 · Full run (no `--real-text`: no Wikipedia data)

In [ ]:
!cd ml/berturk && python train_berturk.py --legal "$LEGAL" --out out_clean 2>&1 | tail -20

In [ ]:
# Save the result: a zip you can download (left panel > Files) and, if you set HF_TOKEN, a PRIVATE upload to Hugging Face.
# Make it public later on the model page (Settings > Visibility) after checking the scores.
import shutil, os
z = shutil.make_archive("katip-mask-base-clean", "zip", "ml/berturk/out_clean")
print("zip:", z, round(os.path.getsize(z) / 1e6, 1), "MB")
try:
    from google.colab import files; files.download(z)
except Exception:
    pass
UPLOAD = False   # set True to upload
if UPLOAD and HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo("f1k/katip-mask-base-clean", private=True, exist_ok=True)
    api.upload_folder(folder_path="ml/berturk/out_clean", repo_id="f1k/katip-mask-base-clean", commit_message="trained in a notebook (katip-mask-base-clean)")
    print("uploaded to https://huggingface.co/f1k/katip-mask-base-clean (private)")

**Next:** send the scores back in the Katip thread; we score it on KVKK-Bench next to Base v2 and decide on Apache-2.0.